# LAB 2 · ADF → ARIMA / SARIMA
### Notebook khung cho nhóm 6 người
**Hạn hoàn tất: 26/10/2026** · Leader: **TODO tên** · Lớp / nhóm: **TODO**

> Bản khung, chưa phải bài giải hoàn chỉnh. Code đọc dữ liệu, EDA và các bước kiểm tra có thể chạy ngay.
> Mô hình tắt mặc định; nhóm phải chọn tham số rồi bật `RUN_MODELS`.
> Mọi ô **TODO** cần được điền bằng nhận xét của nhóm từ kết quả thực chạy.

**Mục lục**
0. Phân công, lịch và thiết lập chung
1. Bài 1 — Khám phá dữ liệu
2. Bài 2 — ADF chuỗi gốc
3. Bài 3 — Sai phân và ADF lần hai
4. Bài 4 — ACF/PACF, đề xuất tham số
5. Bài 5 — ARIMA: fit, phần dư, forecast, đánh giá
6. Nâng cao — SARIMA trên Dataset 2, so với ARIMA
7. Kết luận và checklist bàn giao

## 0. Phân công và lịch nhóm

| Người | Việc chính | Đầu ra / bàn giao |
|---|---|---|
| Leader - TODO tên | Bài 1, EDA cả hai dataset; notebook chung; tổng hợp | Dữ liệu chuẩn, EDA, kết luận và bản cuối |
| TV2 - TODO tên | Bài 2-3; ADF và sai phân cả hai dataset | Bảng ADF, đồ thị sai phân, đề xuất d/D |
| TV3 - TODO tên | Bài 4; ACF/PACF cả hai dataset | Đồ thị, giải thích p/q/P/Q, cấu hình ứng viên |
| TV4 - TODO tên | Bài 5.1-3; fit ARIMA, AIC/BIC, phần dư; ARIMA dataset 2 | Mô hình, diagnostics, câu hỏi 1-5 |
| TV5 - TODO tên | Bài 5.4-5; forecast, metrics, bảng so sánh dataset 2 | Đồ thị, MAE/RMSE/MAPE, câu hỏi 6-9 |
| TV6 - TODO tên | SARIMA nâng cao dataset 2 | Fit, forecast, MAE/RMSE, nhận xét so với ARIMA |

| Mốc nội bộ đề xuất | Công việc |
|---|---|
| 22/10 | Điền tên, nhận việc, thống nhất dữ liệu và cách đặt biến |
| 23/10 - 20h | Leader/TV2/TV3 bàn giao EDA, ADF, ACF/PACF |
| 24/10 - 20h | Mọi người gửi code chạy được, hình/bảng và nhận xét |
| 25/10 | Kiểm tra chéo TV2-TV3, TV4-TV5, Leader-TV6; ghép bài |
| 26/10 | Restart Kernel + Run All; chốt notebook và báo cáo |

**Quy ước**
- Mỗi người nộp code + hình/bảng + nhận xét + câu trả lời phần mình.
- Giữ các tên biến chung: `y1`, `train1`, `test1`, `y2`, `train2`, `test2`.
- Chia 80/20 theo thời gian, không shuffle. Chuỗi “gốc” trong bước ADF/ACF/PACF là **train chưa sai phân**.
- Chọn tham số trên train; không dùng test để thử và chọn mô hình. Muốn tinh chỉnh thêm thì dùng validation theo thời gian trong train.
- Chỉ so ARIMA với SARIMA trên **cùng Dataset 2**, cùng train/test và thang đo.

### 0.1. Cài môi trường và cách chạy
Mở cả folder `Lab2` trong VS Code, chọn Python kernel cho notebook; hoặc mở bằng JupyterLab.
Trong terminal chạy `python -m pip install -r requirements.txt` một lần.

Chạy từ trên xuống. Để thử mô hình: điền cấu hình tại ô `CẤU HÌNH NHÓM`, đổi `RUN_MODELS = True`, rồi **Restart Kernel → Run All**.
CSV đã nằm trong `data/`, notebook không cần tải dữ liệu lúc chạy.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython import get_ipython
from IPython.display import display
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox

shell = get_ipython()
if shell is not None:
    shell.run_line_magic('matplotlib', 'inline')
plt.rcParams.update({'figure.figsize': (11, 4), 'axes.grid': True, 'font.size': 10})
pd.set_option('display.max_columns', 16)

In [ ]:
# CẤU HÌNH NHÓM — điền sau khi xem ADF và ACF/PACF trên train.
LAB2_SCAFFOLD_VERSION = 1
PROJECT_ROOT_OVERRIDE = None  # Ví dụ: Path(r'D:/mon_hoc/Lab2') nếu cần.
candidates = [Path.cwd(), Path.cwd() / 'Lab2', Path.cwd() / 'outputs' / 'Lab2']
if PROJECT_ROOT_OVERRIDE is not None:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).resolve()
else:
    PROJECT_ROOT = next((p.resolve() for p in candidates if (p / 'data' / 'a10.csv').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Mở notebook từ folder Lab2, hoặc điền PROJECT_ROOT_OVERRIDE.')

DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_ROOT = Path(os.environ.get('LAB2_OUTPUT_ROOT', str(PROJECT_ROOT)))
FIGURES_DIR = OUTPUT_ROOT / 'figures'
TABLES_DIR = OUTPUT_ROOT / 'tables'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_RATIO = 0.80
ALPHA = 0.05
SEASON_LENGTH = 12
RUN_MODELS = False
ARIMA_ORDERS = []            # TODO TV3/TV4: 1-2 bộ (p,d,q), ví dụ [(1,1,1), (2,1,1)].
ARIMA_ORDER_DS2 = None        # TODO TV3/TV4: ARIMA baseline trên Dataset 2.
SARIMA_ORDER = None           # TODO TV3/TV6: (p,d,q).
SARIMA_SEASONAL_ORDER = None  # TODO TV3/TV6: (P,D,Q,12).
print('Folder:', PROJECT_ROOT)
print('RUN_MODELS =', RUN_MODELS, '| hạn nhóm: 26/10/2026')

### 0.2. Hàm dùng chung
Các hàm chỉ hỗ trợ tính toán/vẽ hình. Nhóm vẫn cần giải thích kết quả.
`MAPE_pct` là phần trăm; nếu giá trị thực bằng 0, hàm trả `NaN` cho MAPE và vẫn tính MAE/RMSE.

In [ ]:
def load_monthly(filename, date_col, value_col):
    frame = pd.read_csv(DATA_DIR / filename)
    frame[date_col] = pd.to_datetime(frame[date_col], errors='raise')
    frame[value_col] = pd.to_numeric(frame[value_col], errors='raise')
    if frame[date_col].duplicated().any():
        raise ValueError(f'{filename}: trùng mốc thời gian.')
    series = frame.set_index(date_col)[value_col].sort_index().astype(float)
    monthly = series.asfreq('MS')
    if len(monthly) != len(series) or monthly.isna().any() or not np.isfinite(monthly).all():
        raise ValueError(f'{filename}: kiểm tra tháng thiếu, ngày không đúng đầu tháng hoặc giá trị thiếu.')
    return monthly.rename(value_col)

def time_split(series, ratio=TRAIN_RATIO):
    cut = int(len(series) * ratio)
    if not 0 < cut < len(series):
        raise ValueError('Train và test phải đều có quan sát.')
    train, test = series.iloc[:cut].copy(), series.iloc[cut:].copy()
    assert train.index.max() < test.index.min()
    return train, test

def adf_result(series, label):
    clean = series.dropna()
    stat, pvalue, used_lag, nobs, critical, icbest = adfuller(clean, autolag='AIC')
    return pd.Series({
        'chuoi': label, 'ADF_statistic': stat, 'p_value': pvalue,
        'used_lag': used_lag, 'nobs': nobs, 'critical_1pct': critical['1%'],
        'critical_5pct': critical['5%'], 'critical_10pct': critical['10%'],
        'bac_bo_H0_5pct': bool(pvalue < ALPHA),
    })

def correlograms(series, title, filename):
    clean = series.dropna()
    lags = min(36, len(clean) // 2 - 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    plot_acf(clean, lags=lags, ax=axes[0], zero=False)
    plot_pacf(clean, lags=lags, ax=axes[1], method='ywm', zero=False)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / filename, dpi=150, bbox_inches='tight')
    plt.show()

def metrics(actual, predicted):
    if not actual.index.equals(predicted.index):
        raise ValueError('Forecast phải khớp index test.')
    truth, pred = actual.to_numpy(dtype=float), predicted.to_numpy(dtype=float)
    if len(truth) == 0 or not (np.isfinite(truth).all() and np.isfinite(pred).all()):
        raise ValueError('Dữ liệu đánh giá phải hữu hạn và khác rỗng.')
    error = truth - pred
    return {
        'MAE': float(np.mean(np.abs(error))),
        'RMSE': float(np.sqrt(np.mean(error ** 2))),
        'MAPE_pct': float(np.mean(np.abs(error / truth)) * 100) if np.all(truth != 0) else np.nan,
    }

def forecast_series(fitted, test):
    forecast = fitted.get_forecast(steps=len(test))
    # Kiểm tra index thật, không gán lại index để che sai lệch thời gian.
    predicted = forecast.predicted_mean.rename('Forecast')
    assert predicted.index.equals(test.index), 'Kiểm tra tần suất và horizon dự báo.'
    interval = forecast.conf_int(alpha=ALPHA)
    return predicted, interval

def plot_forecast(train, test, predicted, interval, title, filename):
    fig, ax = plt.subplots(figsize=(12, 4))
    train.plot(ax=ax, label='Train')
    test.plot(ax=ax, label='Test thực tế', color='black')
    predicted.plot(ax=ax, label='Forecast', color='tab:orange')
    ax.fill_between(test.index, interval.iloc[:, 0], interval.iloc[:, 1], alpha=.18, color='tab:orange', label='Khoảng dự báo 95%')
    ax.axvline(test.index[0], color='gray', linestyle='--', alpha=.6)
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / filename, dpi=150, bbox_inches='tight')
    plt.show()

def residual_diagnostics(fitted, label, filename, model_df=0):
    burn = int(getattr(fitted, 'loglikelihood_burn', 0))
    residuals = fitted.resid.iloc[burn:].dropna()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    residuals.plot(ax=axes[0], title=f'Phần dư - {label}')
    axes[0].axhline(0, color='black', linestyle='--')
    plot_acf(residuals, lags=min(24, len(residuals)//2 - 1), ax=axes[1], zero=False)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / filename, dpi=150, bbox_inches='tight')
    plt.show()
    print('Mean phần dư:', float(residuals.mean()))
    valid_lags = [k for k in (12, 24) if model_df < k < len(residuals)]
    if valid_lags:
        display(acorr_ljungbox(residuals, lags=valid_lags, model_df=model_df, return_df=True))
    return residuals

## 1. Bài 1 — Khám phá dữ liệu 
**Mục tiêu:** load CSV, chuyển date thành index tháng, vẽ chuỗi và nhận diện xu hướng/mùa vụ.
Dataset 1 dùng đúng `a10.csv` trong đề (cột `date`, `value`). Xác nhận lại tên nghiệp vụ và đơn vị với giảng viên trước khi ghi báo cáo.

In [ ]:
y1 = load_monthly('a10.csv', 'date', 'value')
y2 = load_monthly('airline-passengers.csv', 'Month', 'Passengers')
train1, test1 = time_split(y1)
train2, test2 = time_split(y2)
data_summary = pd.DataFrame([
    {'dataset': 'Dataset 1', 'n': len(y1), 'train': len(train1), 'test': len(test1), 'tu': y1.index.min(), 'den': y1.index.max()},
    {'dataset': 'Dataset 2', 'n': len(y2), 'train': len(train2), 'test': len(test2), 'tu': y2.index.min(), 'den': y2.index.max()},
])
display(data_summary)
display(train1.head().to_frame())
data_summary.to_csv(TABLES_DIR / 'data_summary.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
train1.plot(ax=ax, label='Dataset 1 - train')
train1.rolling(12).mean().plot(ax=ax, label='Trung bình trượt 12 tháng')
ax.set_title('Dataset 1: chuỗi gốc trên train')
ax.set_ylabel('value (xác nhận đơn vị theo nguồn)')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'ds1_eda.png', dpi=150, bbox_inches='tight')
plt.show()
display(train1.describe().to_frame())

### Nhận xét Bài 1 · 
1. Chuỗi có xu hướng không? **TODO: chiều hướng, giai đoạn và bằng chứng từ đồ thị.**
2. Có dấu hiệu mùa vụ không? **TODO: chu kỳ nghi ngờ và điểm lặp lại.**
3. Chuỗi có khả năng dừng không? **TODO: nhận định sơ bộ, chưa thay cho kiểm định.**

## 2. Bài 2 — ADF trên chuỗi gốc · TV2
Kiểm định trên `train1` chưa sai phân. Ghi rõ `autolag='AIC'`, mức ý nghĩa 5% và giả thuyết sử dụng.

In [ ]:
adf1_original = adf_result(train1, 'DS1 train gốc')
display(adf1_original.to_frame('ket_qua'))

### Trả lời Bài 2 · TV2 điền
- H₀ và H₁ của ADF: **TODO**.
- ADF statistic, p-value và mức ý nghĩa: **TODO từ kết quả thực chạy**.
- p-value cho biết điều gì? **TODO**.
- Kết luận: **TODO**; dùng “bác bỏ / chưa đủ bằng chứng bác bỏ H₀”, không coi p-value là xác suất H₀ đúng.

## 3. Bài 3 — Sai phân bậc 1 và ADF lại · TV2
Bước dưới làm sai phân bậc 1 theo yêu cầu đề. Giá trị `d` cuối cùng phải dựa trên kiểm định và đồ thị, không mặc định từ ví dụ.

In [ ]:
train1_diff1 = train1.diff().dropna()
fig, ax = plt.subplots(figsize=(12, 4))
train1_diff1.plot(ax=ax, title='Dataset 1: sai phân bậc 1 trên train')
ax.axhline(0, color='black', linestyle='--')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'ds1_diff1.png', dpi=150, bbox_inches='tight')
plt.show()
adf1_diff1 = adf_result(train1_diff1, 'DS1 train sai phân bậc 1')
adf_table1 = pd.DataFrame([adf1_original, adf1_diff1])
display(adf_table1)
adf_table1.to_csv(TABLES_DIR / 'ds1_adf.csv', index=False)

### Trả lời Bài 3 · TV2 điền
- Sau sai phân bậc 1, chuỗi đã có bằng chứng dừng chưa? **TODO**.
- Chọn `d = TODO` vì **TODO**.
- Nếu bằng chứng chưa đủ: **TODO bước kiểm tra tiếp theo; không ép kết luận dừng**.

## 4. Bài 4 — ACF / PACF · TV3
Ứng viên hiện tại là chuỗi sai phân bậc 1. Nhóm đổi `stationary1` nếu kết quả Bài 3 cho thấy cần xử lý khác.
Các lag nổi bật chỉ là gợi ý cấu hình, không phải tham số được chốt tự động.

In [ ]:
stationary1 = train1_diff1  # TODO TV2/TV3 xác nhận phép biến đổi phù hợp.
if bool(adf_result(stationary1, 'DS1 ứng viên')['bac_bo_H0_5pct']):
    correlograms(stationary1, 'Dataset 1: ACF/PACF của chuỗi ứng viên dừng', 'ds1_acf_pacf.png')
else:
    print('ADF chưa ủng hộ chuỗi ứng viên dừng ở mức 5%; TV2/TV3 cần rà soát trước khi chọn p, q.')

### Trả lời Bài 4 · TV3 điền
- ACF cắt cụt / suy giảm tại lag nào? **TODO**.
- PACF cắt cụt / suy giảm tại lag nào? **TODO**.
- Có đỉnh mùa vụ tại lag 12, 24... không? **TODO**.
- Đề xuất 1–2 bộ `(p,d,q)` và giải thích: **TODO**.
- Sau khi chốt, cập nhật `ARIMA_ORDERS` ở ô cấu hình chung.

## 5. Bài 5 — ARIMA trên Dataset 1
### 5.1. Fit và so sánh AIC/BIC · TV4
Chỉ fit trên `train1` gốc; thư viện tự sai phân theo `d` trong mô hình.
Không đưa `train1_diff1` vào ARIMA khi `d > 0` vì sẽ sai phân hai lần.
Điền `ARIMA_ORDERS` trước khi bật `RUN_MODELS`; chọn theo tiêu chí trong train, không nhìn test để chọn.

In [ ]:
fitted1 = {}
model_comparison1 = pd.DataFrame()
best_order1 = None
best_model1 = None
if RUN_MODELS:
    if not 1 <= len(ARIMA_ORDERS) <= 2:
        raise ValueError('TV3/TV4 cần điền 1-2 cấu hình ARIMA_ORDERS.')
    rows = []
    for order in ARIMA_ORDERS:
        fitted = ARIMA(train1, order=tuple(order)).fit()
        fitted1[tuple(order)] = fitted
        rows.append({'order': str(tuple(order)), 'AIC': fitted.aic, 'BIC': fitted.bic,
                     'converged': bool(fitted.mle_retvals.get('converged', False))})
    model_comparison1 = pd.DataFrame(rows).sort_values('AIC')
    display(model_comparison1)
    model_comparison1.to_csv(TABLES_DIR / 'ds1_arima_aic_bic.csv', index=False)
    eligible = {o: f for o, f in fitted1.items() if bool(f.mle_retvals.get('converged', False))}
    if not eligible:
        raise RuntimeError('Các cấu hình chưa hội tụ; TV4 cần kiểm tra trước khi dự báo.')
    best_order1 = min(eligible, key=lambda o: eligible[o].aic)
    best_model1 = eligible[best_order1]
    print('Ứng viên AIC nhỏ nhất trong các mô hình hội tụ:', best_order1)
    print(best_model1.summary())
else:
    print('TODO TV3/TV4: điền cấu hình, bật RUN_MODELS rồi chạy lại từ đầu.')

### 5.2. Chẩn đoán phần dư · TV4
Kiểm tra đồ thị phần dư, mean, ACF và Ljung–Box. p-value lớn không tự chứng minh phần dư là white noise.
Nếu phần dư còn cấu trúc, quay lại xử lý dữ liệu / cấu hình trong train trước khi chốt đánh giá test.

In [ ]:
residuals1 = None
if best_model1 is not None:
    residuals1 = residual_diagnostics(best_model1, f'ARIMA{best_order1}', 'ds1_residuals.png',
                                      model_df=best_order1[0] + best_order1[2])
else:
    print('Chưa có mô hình ARIMA để kiểm tra phần dư.')

### Trả lời Bài 5, câu 1–5 · TV4 điền
1. Vì sao không được shuffle dữ liệu chuỗi thời gian? Test set dùng làm gì? **TODO**.
2. Ý nghĩa `p`, `d`, `q` đã chọn và lý do chọn? **TODO**.
3. Mô hình có hội tụ không? Dẫn kết quả `converged`. **TODO**.
4. Phần dư có phù hợp giả định white noise không? Dẫn mean, ACF, Ljung–Box. **TODO**.
5. Nếu phần dư còn tự tương quan thì điều chỉnh gì? **TODO**.

### 5.3. Forecast và đánh giá trên test · TV5
Tính MAE, RMSE và MAPE (%); forecast đúng số tháng trong test, không fit lại bằng test.

In [ ]:
forecast1 = None
metrics1 = None
if best_model1 is not None:
    forecast1, interval1 = forecast_series(best_model1, test1)
    plot_forecast(train1, test1, forecast1, interval1, f'Dataset 1 - ARIMA{best_order1}', 'ds1_forecast.png')
    metrics1 = pd.DataFrame([{'model': f'ARIMA{best_order1}', **metrics(test1, forecast1)}])
    display(metrics1)
    metrics1.to_csv(TABLES_DIR / 'ds1_metrics.csv', index=False)
    errors1 = pd.DataFrame({'actual': test1, 'forecast': forecast1, 'abs_error': (test1 - forecast1).abs()})
    display(errors1.nlargest(3, 'abs_error'))
    errors1.to_csv(TABLES_DIR / 'ds1_predictions.csv', index_label='date')
else:
    print('TODO TV5: nhận mô hình từ TV4 sau khi chạy phần fit.')

### Trả lời Bài 5, câu 6–9 · TV5 điền
6. Forecast có bám sát xu hướng / mùa vụ không? **TODO**.
7. Sai lệch lớn nhất ở tháng nào, trị số bao nhiêu, nguyên nhân giả định? **TODO**.
8. Chỉ số nào nhạy với outlier nhất? Giải thích bằng công thức. **TODO**.
9. RMSE lớn / nhỏ có ý nghĩa gì trong đơn vị và quy mô dữ liệu? **TODO**.

**Đánh giá chung Dataset 1:** ưu điểm **TODO**; hạn chế **TODO**; hướng cải thiện **TODO**.

## 6. Nâng cao — SARIMA trên Dataset 2
### 6.1. EDA · Leader; ADF / sai phân · TV2
Dataset 2 có dữ liệu hành khách theo tháng. `s = 12` theo đề.
Sai phân thường bậc 1 + sai phân mùa vụ bậc 1 là **gợi ý để kiểm tra**, không tự động chốt `d = D = 1` cho mọi cấu hình.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
train2.plot(ax=ax, title='Dataset 2: chuỗi gốc trên train')
ax.set_ylabel('Passengers')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'ds2_eda.png', dpi=150, bbox_inches='tight')
plt.show()
train2_diff1 = train2.diff().dropna()
train2_diff1_seasonal = train2.diff().diff(SEASON_LENGTH).dropna()
adf_table2 = pd.DataFrame([
    adf_result(train2, 'DS2 train gốc'),
    adf_result(train2_diff1, 'DS2 sai phân thường bậc 1'),
    adf_result(train2_diff1_seasonal, 'DS2 sai phân thường 1 + mùa vụ 1'),
])
display(adf_table2)
adf_table2.to_csv(TABLES_DIR / 'ds2_adf.csv', index=False)
fig, ax = plt.subplots(figsize=(12, 4))
train2_diff1_seasonal.plot(ax=ax, title='Dataset 2: ứng viên sau sai phân thường + mùa vụ')
ax.axhline(0, color='black', linestyle='--')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'ds2_differences.png', dpi=150, bbox_inches='tight')
plt.show()

### Nhận xét · Leader / TV2 điền
- Dấu hiệu mùa vụ trong đồ thị: **TODO**.
- Kết quả ADF gốc và sau các bước sai phân: **TODO**.
- `d`, `D` đề xuất và bằng chứng: **TODO**.
- Có cần xem xét biến đổi ổn định phương sai không? **TODO**. Nếu biến đổi, xử lý hoàn nguyên trước khi đánh giá ở thang gốc.

### 6.2. ACF/PACF thường và mùa vụ · TV3

In [ ]:
stationary2 = train2_diff1_seasonal  # TODO xác nhận sau ADF.
if bool(adf_result(stationary2, 'DS2 ứng viên')['bac_bo_H0_5pct']):
    correlograms(stationary2, 'Dataset 2: ACF/PACF sau xử lý ứng viên', 'ds2_acf_pacf.png')
else:
    print('Ứng viên Dataset 2 chưa có bằng chứng dừng theo ADF 5%; TV2/TV3 cần rà soát.')

### Chọn tham số · TV3 / TV6 điền
- Lag thường nổi bật: **TODO** → gợi ý `p, q`: **TODO**.
- Lag mùa vụ 12, 24, ...: **TODO** → gợi ý `P, Q`: **TODO**.
- SARIMA `(p,d,q)(P,D,Q,12)`: **TODO**; cập nhật cấu hình chung.
- ARIMA baseline trên chính Dataset 2: **TODO**; TV4 cập nhật `ARIMA_ORDER_DS2`.

### 6.3. ARIMA baseline · TV4; SARIMA · TV6
Hai mô hình đều fit trên `train2` gốc và forecast đúng `test2`.
Truyền `order`/`seasonal_order` vào mô hình thay vì đưa chuỗi đã sai phân vào lần nữa.

In [ ]:
arima2 = None
sarima2 = None
if RUN_MODELS:
    if any(value is None for value in (ARIMA_ORDER_DS2, SARIMA_ORDER, SARIMA_SEASONAL_ORDER)):
        raise ValueError('Điền đầy đủ cấu hình ARIMA baseline và SARIMA của Dataset 2.')
    if len(SARIMA_SEASONAL_ORDER) != 4 or SARIMA_SEASONAL_ORDER[3] != SEASON_LENGTH:
        raise ValueError('SARIMA_SEASONAL_ORDER phải có dạng (P,D,Q,12).')
    arima2 = ARIMA(train2, order=tuple(ARIMA_ORDER_DS2)).fit()
    sarima2 = SARIMAX(train2, order=tuple(SARIMA_ORDER),
                      seasonal_order=tuple(SARIMA_SEASONAL_ORDER)).fit(disp=False, maxiter=200)
    comparison_fit2 = pd.DataFrame([
        {'model': 'ARIMA', 'order': str(ARIMA_ORDER_DS2), 'seasonal_order': '-',
         'AIC': arima2.aic, 'BIC': arima2.bic, 'converged': bool(arima2.mle_retvals.get('converged', False))},
        {'model': 'SARIMA', 'order': str(SARIMA_ORDER), 'seasonal_order': str(SARIMA_SEASONAL_ORDER),
         'AIC': sarima2.aic, 'BIC': sarima2.bic, 'converged': bool(sarima2.mle_retvals.get('converged', False))},
    ])
    display(comparison_fit2)
    comparison_fit2.to_csv(TABLES_DIR / 'ds2_fit.csv', index=False)
    if not bool(comparison_fit2['converged'].all()):
        raise RuntimeError('Dataset 2 còn mô hình chưa hội tụ; kiểm tra trước khi đánh giá.')
    residual_diagnostics(sarima2, 'SARIMA Dataset 2', 'ds2_sarima_residuals.png',
                         model_df=SARIMA_ORDER[0] + SARIMA_ORDER[2] + SARIMA_SEASONAL_ORDER[0] + SARIMA_SEASONAL_ORDER[2])
else:
    print('TODO TV4/TV6: điền cấu hình và bật RUN_MODELS để fit hai mô hình Dataset 2.')

### 6.4. Forecast và bảng so sánh Dataset 2 · TV5 / TV6

In [ ]:
forecast_arima2 = None
forecast_sarima2 = None
comparison_metrics2 = pd.DataFrame()
if arima2 is not None and sarima2 is not None:
    forecast_arima2, ci_arima2 = forecast_series(arima2, test2)
    forecast_sarima2, ci_sarima2 = forecast_series(sarima2, test2)
    plot_forecast(train2, test2, forecast_sarima2, ci_sarima2, 'Dataset 2 - SARIMA', 'ds2_sarima_forecast.png')
    comparison_metrics2 = pd.DataFrame([
        {'model': 'ARIMA', **metrics(test2, forecast_arima2)},
        {'model': 'SARIMA', **metrics(test2, forecast_sarima2)},
    ])[['model', 'MAE', 'RMSE']]
    display(comparison_metrics2)
    comparison_metrics2.to_csv(TABLES_DIR / 'ds2_comparison_metrics.csv', index=False)
    fig, ax = plt.subplots(figsize=(12, 4))
    test2.plot(ax=ax, label='Test thực tế', color='black')
    forecast_arima2.plot(ax=ax, label='ARIMA')
    forecast_sarima2.plot(ax=ax, label='SARIMA')
    ax.set_title('Dataset 2: so sánh trên cùng test và thang đo')
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'ds2_compare_forecasts.png', dpi=150, bbox_inches='tight')
    plt.show()
    pd.DataFrame({'actual': test2, 'ARIMA': forecast_arima2, 'SARIMA': forecast_sarima2}).to_csv(
        TABLES_DIR / 'ds2_predictions.csv', index_label='date')
else:
    print('Chưa có cặp mô hình Dataset 2 để so sánh.')

### Kết luận nâng cao · TV6 viết, TV5 đối chiếu số liệu
- SARIMA có cải thiện MAE/RMSE so với ARIMA trên Dataset 2 không? **TODO số liệu**.
- Forecast có theo được mùa vụ không? **TODO minh chứng đồ thị**.
- Mô hình hội tụ / phần dư còn cấu trúc không? **TODO**.
- Hạn chế, giả định và hướng cải thiện: **TODO**.

## 7. Kết luận chung và checklist · Leader
### Tóm tắt cuối bài
1. Dataset 1: chuỗi gốc **TODO**, phép xử lý **TODO**, cấu hình cuối **TODO**.
2. Kết quả dự báo Dataset 1: MAE **TODO**, RMSE **TODO**, MAPE **TODO %**.
3. Dataset 2: SARIMA **TODO**, so với ARIMA **TODO**.
4. Bài học về chuỗi dừng, sai phân, mùa vụ và đánh giá theo thời gian: **TODO**.

### Checklist trước khi chốt 26/10
- [ ] Điền tên đủ 6 người và thống nhất mức đóng góp.
- [ ] Trả lời đủ câu hỏi Bài 1–5; xác nhận yêu cầu nộp phần nâng cao với giảng viên.
- [ ] Mọi nhận xét có số liệu/đồ thị thực, không giữ nguyên ví dụ tham số như kết luận.
- [ ] Chọn tham số trên train; test không tham gia fit / chọn tham số.
- [ ] Dataset 1 đủ MAE, RMSE, MAPE và kiểm tra phần dư / hội tụ.
- [ ] ARIMA–SARIMA chỉ so sánh trên cùng Dataset 2 và cùng test.
- [ ] Bật RUN_MODELS sau khi điền cấu hình, Restart Kernel + Run All không lỗi.
- [ ] Bỏ toàn bộ TODO; kiểm tra figures/ và tables/ trước khi nộp.

### Nguồn
- Đề gốc: `resources/de_bai_lab2.pdf`.
- [Dataset 1](https://raw.githubusercontent.com/selva86/datasets/master/a10.csv) · [Dataset 2](https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv).
- [statsmodels ADF](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.adfuller.html) · [ARIMA](https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMA.html) · [SARIMAX](https://www.statsmodels.org/stable/generated/statsmodels.tsa.statespace.sarimax.SARIMAX.html).